# 🔥 PySpark Optimization - Advanced Techniques

**Objective**: Optimize PySpark performance for Anti-Fraud Detection System

## 📋 Content:
- Partitioning strategies (hash, range, custom)
- Caching and persistence (MEMORY_AND_DISK_SER)
- Broadcast variables for join optimization
- Spark UI monitoring and bottleneck identification
- Performance benchmarks

## 🚀 Setup and Initialization

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, sum as spark_sum, avg, broadcast
from pyspark.storagelevel import StorageLevel
import time
import pandas as pd

# Create optimized Spark session
spark = SparkSession.builder \
    .appName("PySparkOptimization") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
    .config("spark.sql.shuffle.partitions", "200") \
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer") \
    .config("spark.kryoserializer.buffer.max", "512m") \
    .getOrCreate()

print(f"✅ Spark Session Created")
print(f"Spark Version: {spark.version}")

## 📊 Load Dataset

In [ ]:
# Load PaySim dataset
dataset_path = "../data/PS_20174392719_1491204439457_log.csv"

start_time = time.time()
df = spark.read.csv(dataset_path, header=True, inferSchema=True)
load_time = time.time() - start_time

print(f"📊 Dataset Loaded")
print(f"Rows: {df.count():,}")
print(f"Columns: {len(df.columns)}")
print(f"Load Time: {load_time:.2f}s")
print(f"Partitions: {df.rdd.getNumPartitions()}")

## 🎯 1. Partitioning Strategies

### 1.1 Hash Partitioning (Default)

In [ ]:
# Default hash partitioning
print("\n🔹 HASH PARTITIONING (Default)")
print(f"Current partitions: {df.rdd.getNumPartitions()}")
print(f"Records per partition: {df.count() / df.rdd.getNumPartitions():,.0f}")

### 1.2 Range Partitioning

In [ ]:
# Range partitioning by amount
print("\n🔹 RANGE PARTITIONING by amount")

start_time = time.time()
df_range = df.repartition(100, col("amount"))
range_time = time.time() - start_time

print(f"Partitions after range partitioning: {df_range.rdd.getNumPartitions()}")
print(f"Repartition Time: {range_time:.2f}s")

### 1.3 Custom Partitioning (by transaction type)

In [ ]:
# Custom partitioning by transaction type
print("\n🔹 CUSTOM PARTITIONING by transaction type")

start_time = time.time()
df_custom = df.repartition(col("type"))
custom_time = time.time() - start_time

print(f"Partitions after custom partitioning: {df_custom.rdd.getNumPartitions()}")
print(f"Repartition Time: {custom_time:.2f}s")

# Show distribution
df_custom.groupBy("type").count().show()

## 💾 2. Caching and Persistence Strategies

### 2.1 MEMORY_ONLY Cache

In [ ]:
print("\n🔹 MEMORY_ONLY CACHING")

# Cache in memory only
start_time = time.time()
df.cache()
df.count()  # Force caching
cache_time = time.time() - start_time

print(f"Cache Time: {cache_time:.2f}s")
print(f"Storage Level: MEMORY_ONLY")

# Test cached access
start_time = time.time()
df.count()
cached_access_time = time.time() - start_time
print(f"Cached Access Time: {cached_access_time:.2f}s")

df.unpersist()

### 2.2 MEMORY_AND_DISK_SER (Optimized)

In [ ]:
print("\n🔹 MEMORY_AND_DISK_SER CACHING")

# Cache with serialization and disk fallback
start_time = time.time()
df.persist(StorageLevel.MEMORY_AND_DISK_SER)
df.count()  # Force caching
persist_time = time.time() - start_time

print(f"Persist Time: {persist_time:.2f}s")
print(f"Storage Level: MEMORY_AND_DISK_SER")

# Test cached access
start_time = time.time()
df.count()
persisted_access_time = time.time() - start_time
print(f"Persisted Access Time: {persisted_access_time:.2f}s")

df.unpersist()

### 2.3 Benchmark Comparison

In [ ]:
print("\n📊 CACHING BENCHMARKS")
print("=" * 50)
print(f"MEMORY_ONLY:")
print(f"  - Cache Time: {cache_time:.2f}s")
print(f"  - Access Time: {cached_access_time:.2f}s")
print(f"MEMORY_AND_DISK_SER:")
print(f"  - Persist Time: {persist_time:.2f}s")
print(f"  - Access Time: {persisted_access_time:.2f}s")
print("=" * 50)

## 📡 3. Broadcast Variables for Join Optimization

### 3.1 Create Small Lookup Table

In [ ]:
# Create a small lookup table (e.g., risk scores by transaction type)
risk_scores = spark.createDataFrame([
    ("CASH_IN", 0.1),
    ("CASH_OUT", 0.8),
    ("DEBIT", 0.05),
    ("PAYMENT", 0.02),
    ("TRANSFER", 0.9)
], ["type", "risk_score"])

print("📊 Risk Scores Lookup Table:")
risk_scores.show()
print(f"Size: {risk_scores.count()} rows")

### 3.2 Join WITHOUT Broadcast (Slow)

In [ ]:
print("\n🔹 JOIN WITHOUT BROADCAST (Slow)")

start_time = time.time()
df_joined_slow = df.join(risk_scores, on="type", how="left")
count_slow = df_joined_slow.count()
join_slow_time = time.time() - start_time

print(f"Join Time: {join_slow_time:.2f}s")
print(f"Result Count: {count_slow:,}")

### 3.3 Join WITH Broadcast (Fast)

In [ ]:
print("\n🔹 JOIN WITH BROADCAST (Fast)")

start_time = time.time()
df_joined_fast = df.join(broadcast(risk_scores), on="type", how="left")
count_fast = df_joined_fast.count()
join_fast_time = time.time() - start_time

print(f"Join Time: {join_fast_time:.2f}s")
print(f"Result Count: {count_fast:,}")

### 3.4 Performance Improvement

In [ ]:
improvement = (join_slow_time - join_fast_time) / join_slow_time * 100
speedup = join_slow_time / join_fast_time

print("\n📊 BROADCAST JOIN IMPROVEMENT")
print("=" * 50)
print(f"Without Broadcast: {join_slow_time:.2f}s")
print(f"With Broadcast: {join_fast_time:.2f}s")
print(f"Improvement: {improvement:.1f}%")
print(f"Speedup: {speedup:.1f}x")
print("=" * 50)

## 📈 4. Spark UI Monitoring and Bottleneck Identification

### 4.1 Enable Spark UI

In [ ]:
# Spark UI is available at: http://localhost:4040
print("🔍 SPARK UI MONITORING")
print("=" * 50)
print(f"Spark UI: http://localhost:4040")
print(f"Application ID: {spark.sparkContext.applicationId}")
print(f"Application Name: {spark.sparkContext.appName}")
print("=" * 50)
print("\n📊 Check Spark UI for:")
  - SQL tab: Query execution plans
  - Stages tab: Task distribution
  - Storage tab: Cached data
  - Environment tab: Configuration")

### 4.2 Identify Bottlenecks

In [ ]:
print("\n🔍 BOTTLENECK IDENTIFICATION")

# Check partition skew
print("\n📊 Partition Skew Analysis:")
partition_sizes = df.rdd.mapPartitions(lambda x: [sum(1 for _ in x)]).collect()
print(f"Min partition size: {min(partition_sizes):,}")
print(f"Max partition size: {max(partition_sizes):,}")
print(f"Avg partition size: {sum(partition_sizes)/len(partition_sizes):,.0f}")
print(f"Skew ratio: {max(partition_sizes)/sum(partition_sizes)*len(partition_sizes):.2f}")

if max(partition_sizes) / sum(partition_sizes) * len(partition_sizes) > 2:
    print("⚠️  WARNING: High partition skew detected!")
    print("   Consider: repartitioning, salting keys")
else:
    print("✅ Partitioning is balanced")

### 4.3 Memory Usage Analysis

In [ ]:
print("\n💾 MEMORY USAGE ANALYSIS")

# Get Spark executor memory info
executor_memory = spark.conf.get("spark.executor.memory", "Not set")
driver_memory = spark.conf.get("spark.driver.memory", "Not set")

print(f"Executor Memory: {executor_memory}")
print(f"Driver Memory: {driver_memory}")

# Estimate DataFrame size
df_size_gb = df.count() * len(df.columns) * 8 / (1024**3)  # Rough estimate
print(f"\nEstimated DataFrame Size: {df_size_gb:.2f} GB")

## 🏆 5. Performance Benchmarks Summary

In [ ]:
print("\n🏆 PERFORMANCE BENCHMARKS SUMMARY")
print("=" * 60)
print(f"\n📊 PARTITIONING:")
print(f"  - Hash (default): {df.rdd.getNumPartitions()} partitions")
print(f"  - Range: {range_time:.2f}s repartition")
print(f"  - Custom: {custom_time:.2f}s repartition")

print(f"\n💾 CACHING:")
print(f"  - MEMORY_ONLY: {cache_time:.2f}s cache, {cached_access_time:.2f}s access")
print(f"  - MEMORY_AND_DISK_SER: {persist_time:.2f}s persist, {persisted_access_time:.2f}s access")

print(f"\n📡 BROADCAST JOIN:")
print(f"  - Without broadcast: {join_slow_time:.2f}s")
print(f"  - With broadcast: {join_fast_time:.2f}s")
print(f"  - Speedup: {speedup:.1f}x")

print(f"\n📈 RECOMMENDATIONS:")
if speedup > 1.5:
    print("  ✅ Use broadcast joins for small lookup tables")
if persisted_access_time < cached_access_time:
    print("  ✅ Use MEMORY_AND_DISK_SER for large datasets")
if max(partition_sizes) / sum(partition_sizes) * len(partition_sizes) < 2:
    print("  ✅ Current partitioning is balanced")
else:
    print("  ⚠️  Consider repartitioning to reduce skew")

print("=" * 60)

## 🧹 Cleanup

In [ ]:
# Cleanup
df.unpersist()
df_range.unpersist()
df_custom.unpersist()
df_joined_slow.unpersist()
df_joined_fast.unpersist()
risk_scores.unpersist()

spark.stop()
print("\n✅ Spark session stopped. Optimization complete!")